# Project 1: one event at a time, in Python

You will find the Higgs boson in simulated CMS collisions.
The Higgs decays to two Z bosons, and each Z decays to two leptons (electrons or muons).
The detector sees only the four leptons.

<img src="img/higgs-decay.svg" width="80%" alt="">

**How to work**

- Run each cell with `Shift` + `Enter`, from top to bottom.
- Replace every `...` with your own code.
- Each part ends with a `check`. ✅ means move on. ❌ comes with a hint.
- If you are stuck for more than five minutes, ask an instructor.

In [ ]:
import json
import math

from checks import check

with open("data/higgs_events.json") as f:
    events = json.load(f)

len(events)

`with open(...) as f:` opens the file and closes it again at the end of the block.
`json.load` turns the file into Python lists and dictionaries.
The file holds the first 50,000 events of the simulated sample.

## Part 1: One event

An event is a dictionary of lists of dictionaries.

<img src="img/event-higgs.svg" width="80%" alt="">

In [ ]:
event = events[47]
event["electron"][0]

**Task.** Count the leptons (electrons plus muons) in `event`.
`len(x)` gives the number of items in the list `x`.

In [ ]:
n_leptons = ...
check("1.1", n_leptons)

## Part 2: Four-momentum

A function turns pT, η, φ and mass into energy and momentum.

$$p_x = p_T \cos\phi \qquad p_y = p_T \sin\phi \qquad p_z = p_T \sinh\eta \qquad E = \sqrt{p_x^2 + p_y^2 + p_z^2 + m^2}$$

$p_T$ is the momentum across the beam, $\eta$ measures the angle to the beam, and $\phi$ is the angle around it.

**Tools.** `math.cos(x)`, `math.sin(x)`, `math.sinh(x)`, `math.sqrt(x)`. `x**2` is $x^2$.

**Task.** Complete the function. The first line is done for you.

In [ ]:
def four_momentum(particle):
    px = particle["pt"] * math.cos(particle["phi"])
    py = ...
    pz = ...
    E = ...
    return {"E": E, "px": px, "py": py, "pz": pz}


check("1.2", four_momentum(event["muon"][0]))

## Part 3: The Z boson

A Z boson is the sum of its two leptons.

Energy and momentum are conserved, so the Z carries the summed four-momentum of its two leptons.
Its mass is

$$m = \sqrt{E^2 - p_x^2 - p_y^2 - p_z^2}$$

**Task.** Complete `add` and `mass`, then run the cell to get the mass of the two electrons.

In [ ]:
def add(p, q):
    return {"E": p["E"] + q["E"], "px": ..., "py": ..., "pz": ...}


def mass(p):
    return ...


e1 = four_momentum(event["electron"][0])
e2 = four_momentum(event["electron"][1])
z_ee = add(e1, e2)
check("1.3", mass(z_ee))

## Part 4: The Higgs boson

The Higgs boson is the sum of the two Z bosons.

**Task.** Build the Z from the two muons, then the Higgs from both Z bosons.

In [ ]:
mu1 = ...
mu2 = ...
z_mumu = ...
higgs = ...
check("1.4", mass(higgs))

## Part 5 (optional): Opposite charges

A Z decays to two leptons of opposite charge.

Event 483 has three muons. Which pairs of them could come from a Z?
Count each pair once: (0, 1) but not also (1, 0).

**Tools.** `range(n)` counts 0, 1, ..., n-1.
Two loops, one inside the other, visit every `(i, j)`.
The condition `i < j` keeps each pair once.
`(i, j)` is a tuple: a short list that cannot change.

<img src="img/combinations.svg" width="75%" alt="">

In [ ]:
muons = events[483]["muon"]
pairs = []
for i in range(len(muons)):
    for j in range(len(muons)):
        if i < j and ...:
            pairs.append((i, j))

check("1.5", pairs)

## Part 6: Every event

The same steps in a loop find the Higgs in every event.

Keep the events with exactly two electrons and two muons, where both pairs have opposite charge.
Compute the Higgs mass of each and collect the masses in a list.

`%%time` on the first line of a cell reports how long the cell takes. Note the time: project 2 repeats this step with NumPy.

<img src="img/pipeline.svg" width="70%" alt="">

In [ ]:
%%time
higgs_masses = []
for event in events:
    electrons = event["electron"]
    muons = event["muon"]
    if len(electrons) == 2 and len(muons) == 2:
        opposite_e = ...
        opposite_mu = ...
        if opposite_e and opposite_mu:
            z_ee = ...
            z_mumu = ...
            higgs_masses.append(...)

In [ ]:
check("1.6", higgs_masses)

**Plot.** `hist` counts the masses into 50 bins from 80 to 180 GeV and draws them.
You should see a peak at 125 GeV.

In [ ]:
import hist

h = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
h.fill(higgs_masses)
h.plot();

## Bonus: Z mass windows

Both Z masses must look like a Z.

The 2012 discovery analysis also cut on the two Z masses. A simple version of that cut: the lighter Z mass between 12 and 120 GeV, and the heavier one between 40 and 120 GeV.

**Tools.** `min(a, b)` and `max(a, b)`. `12 < x < 120` checks both limits at once.

**Task.** Copy your loop from Part 6 and add this cut. Collect the masses in `higgs_masses_cut`.

In [ ]:
higgs_masses_cut = []
...

check("1.7", higgs_masses_cut)

In [ ]:
import matplotlib.pyplot as plt

h_cut = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
h_cut.fill(higgs_masses_cut)
h.plot(label="opposite charges")
h_cut.plot(label="and Z mass cuts")
plt.legend();

**Done.** Project 2 repeats Part 6 with NumPy, for every event of the full sample.
Keep this notebook open to compare the run times.